# 04. Mô hình hóa và đánh giá phân cụm

Notebook này thực hiện toàn bộ phần mô hình hóa cho bài toán phân cụm bộ dữ liệu việc làm Việt Nam từ các artifact đã được tạo ở notebook 03.


## 4.1. Mục tiêu

- Nạp lại feature artifacts và metadata từ notebook 03.
- Huấn luyện nhiều cấu hình phân cụm trên không gian SVD và UMAP.
- So sánh nhiều số cụm `k` bằng các metric nội tại phù hợp cho clustering.
- Chọn mô hình tốt nhất bằng rank tổng hợp.
- Gán `cluster_id` cho train/test, phân tích đặc điểm từng cụm và tạo demo trên tập test.

Vì đây là bài toán học không giám sát, notebook **không dùng Accuracy, F1 hay RMSE**. Thay vào đó, notebook dùng các metric nội tại như **Inertia, Silhouette, Davies-Bouldin, Calinski-Harabasz** để đánh giá cấu trúc cụm.


## 4.2. Dữ liệu đầu vào và kiểm soát rò rỉ dữ liệu

Notebook này chỉ đọc lại artifact đã được sinh từ notebook 03:

- `X_train_svd.npy`, `X_test_svd.npy`
- `X_train_umap.npy`, `X_test_umap.npy`
- `train_metadata.csv`, `test_metadata.csv`
- các model/artifact phụ trợ nếu cần

Nguyên tắc kiểm soát rò rỉ dữ liệu:

- Không fit lại TF-IDF, SVD, UMAP trên tập test.
- Tập test chỉ dùng để **gán cụm** bằng mô hình đã fit trên train và để demo 10 mẫu.
- Việc chọn mô hình tốt nhất và chọn số cụm chỉ dựa trên **train**.
- Do dữ liệu lớn, Silhouette, Davies-Bouldin và Calinski-Harabasz được tính trên một **sample cố định** của train thay vì toàn bộ dữ liệu.


In [ ]:
from __future__ import annotations

import json
import os
import tempfile
import time
from pathlib import Path

import joblib
os.environ.setdefault("MPLCONFIGDIR", str((Path(tempfile.gettempdir()) / "codex_matplotlib").resolve()))
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from IPython.display import Markdown, display


In [ ]:
BASE_DIR = Path("..")
if not (BASE_DIR / "artifacts").exists():
    BASE_DIR = Path.cwd().parent

FEATURES_DIR = BASE_DIR / "artifacts" / "features"
CLUSTERING_DIR = BASE_DIR / "artifacts" / "clustering"
FIGURES_DIR = CLUSTERING_DIR / "figures"

RANDOM_STATE = 42
DEFAULT_K_VALUES = [3, 4, 5, 6, 7, 8, 10, 12]
QUICK_MODE = False
QUICK_K_VALUES = [3, 4]
K_VALUES = QUICK_K_VALUES if QUICK_MODE else DEFAULT_K_VALUES

EVAL_SAMPLE_SIZE = 10000
VIS_SAMPLE_SIZE = 20000
HEAVY_KMEANS_SVD_SAMPLE_SIZE = 150000
HEAVY_KMEANS_UMAP_SAMPLE_SIZE = None
PRESERVE_EXISTING_MANUAL_LABELS = True

CLUSTERING_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR.resolve())
print("FEATURES_DIR:", FEATURES_DIR.resolve())
print("CLUSTERING_DIR:", CLUSTERING_DIR.resolve())
print("K_VALUES:", K_VALUES)
print("QUICK_MODE:", QUICK_MODE)


## 4.3. Mô hình được lựa chọn

Notebook huấn luyện tối thiểu 3 cấu hình:

1. `KMeans` trên `X_train_svd.npy`
2. `MiniBatchKMeans` trên `X_train_svd.npy`
3. `KMeans` trên `X_train_umap.npy`

Với dữ liệu train khoảng hơn 500 nghìn mẫu, `KMeans` thường trên không gian SVD 300 chiều có thể rất nặng. Vì vậy notebook có cơ chế **fit trên sample train đại diện** cho cấu hình nặng nếu số dòng vượt ngưỡng, sau đó dùng model để dự đoán trên toàn bộ train/test. Cấu hình `MiniBatchKMeans` trên SVD vẫn được ưu tiên như cấu hình ổn định nhất cho dữ liệu lớn.


In [ ]:
def load_artifacts(base_dir: Path) -> dict[str, object]:
    feature_dir = base_dir / "artifacts" / "features"
    required_files = {
        "train_svd": feature_dir / "X_train_svd.npy",
        "test_svd": feature_dir / "X_test_svd.npy",
        "train_umap": feature_dir / "X_train_umap.npy",
        "test_umap": feature_dir / "X_test_umap.npy",
        "train_metadata": feature_dir / "train_metadata.csv",
        "test_metadata": feature_dir / "test_metadata.csv",
        "test_features": feature_dir / "X_test_features.npz",
        "tfidf_model": feature_dir / "tfidf_model.joblib",
    }
    missing = [str(path) for path in required_files.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Thiếu artifact đầu vào từ notebook 03: " + ", ".join(missing))

    train_svd = np.load(required_files["train_svd"], mmap_mode="r")
    test_svd = np.load(required_files["test_svd"], mmap_mode="r")
    train_umap = np.load(required_files["train_umap"], mmap_mode="r")
    test_umap = np.load(required_files["test_umap"], mmap_mode="r")

    train_metadata = pd.read_csv(required_files["train_metadata"])
    test_metadata = pd.read_csv(required_files["test_metadata"])

    if train_svd.shape[0] != len(train_metadata):
        raise ValueError("Số dòng X_train_svd không khớp train_metadata.")
    if test_svd.shape[0] != len(test_metadata):
        raise ValueError("Số dòng X_test_svd không khớp test_metadata.")
    if train_umap.shape[0] != len(train_metadata):
        raise ValueError("Số dòng X_train_umap không khớp train_metadata.")
    if test_umap.shape[0] != len(test_metadata):
        raise ValueError("Số dòng X_test_umap không khớp test_metadata.")

    train_features_path = feature_dir / "X_train_features.npz"
    train_features = sparse.load_npz(train_features_path) if train_features_path.exists() else None
    test_features = sparse.load_npz(required_files["test_features"])

    tfidf_model = joblib.load(required_files["tfidf_model"])

    feature_summary_path = feature_dir / "feature_engineering_summary.csv"
    feature_summary = pd.read_csv(feature_summary_path) if feature_summary_path.exists() else None

    return {
        "feature_dir": feature_dir,
        "train_svd": train_svd,
        "test_svd": test_svd,
        "train_umap": train_umap,
        "test_umap": test_umap,
        "train_metadata": train_metadata,
        "test_metadata": test_metadata,
        "train_features": train_features,
        "test_features": test_features,
        "tfidf_model": tfidf_model,
        "feature_summary": feature_summary,
    }


def make_eval_sample(n_rows: int, sample_size: int, random_state: int = 42) -> np.ndarray:
    rng = np.random.default_rng(random_state)
    actual_size = min(sample_size, n_rows)
    return np.sort(rng.choice(n_rows, size=actual_size, replace=False))


def top_values(series: pd.Series, top_n: int = 5) -> str:
    cleaned = series.fillna("unknown").astype(str).str.strip()
    cleaned = cleaned.replace("", "unknown")
    counts = cleaned.value_counts().head(top_n)
    if counts.empty:
        return ""
    return "; ".join(f"{value} ({count})" for value, count in counts.items())


def first_value_from_top_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    return text.split(";")[0].rsplit(" (", 1)[0].strip()


def build_suggested_label(row: pd.Series) -> str:
    industry = first_value_from_top_text(row.get("top_job_industry", ""))
    title = first_value_from_top_text(row.get("top_job_title", ""))
    experience = first_value_from_top_text(row.get("top_experience_level", ""))
    if industry and title:
        return f"{industry} - {title}"
    if industry and experience:
        return f"{industry} - {experience}"
    return industry or title or experience or f"cluster_{int(row['cluster_id'])}"


def get_feature_matrix(artifacts: dict[str, object], feature_space: str, split: str):
    key = f"{split}_{feature_space}"
    return artifacts[key]


def fit_with_optional_sampling(X, estimator, fit_sample_size: int | None, random_state: int = 42):
    fit_note = "fit_full_train"
    fit_rows = X.shape[0]
    if fit_sample_size is not None and X.shape[0] > fit_sample_size:
        fit_idx = make_eval_sample(X.shape[0], fit_sample_size, random_state=random_state)
        X_fit = X[fit_idx]
        fit_note = f"fit_on_sample_{len(fit_idx)}"
        fit_rows = len(fit_idx)
    else:
        fit_idx = None
        X_fit = X

    estimator.fit(X_fit)
    return estimator, fit_idx, fit_note, fit_rows


def evaluate_clustering(
    estimator_cls,
    estimator_params: dict[str, object],
    X_train,
    X_eval,
    eval_idx: np.ndarray,
    model_name: str,
    feature_space: str,
    k: int,
    fit_sample_size: int | None = None,
) -> dict[str, object]:
    estimator = estimator_cls(**estimator_params, n_clusters=k)
    start = time.perf_counter()
    estimator, fit_idx, fit_note, fit_rows = fit_with_optional_sampling(
        X_train,
        estimator,
        fit_sample_size=fit_sample_size,
        random_state=estimator_params.get("random_state", RANDOM_STATE),
    )
    labels_full = estimator.predict(X_train)
    train_time_sec = time.perf_counter() - start

    labels_sample = labels_full[eval_idx]
    unique_labels = np.unique(labels_sample)
    if len(unique_labels) < 2:
        raise ValueError(f"Mô hình {model_name} - {feature_space} - k={k} sinh chưa đủ cụm trên sample đánh giá.")

    cluster_sizes = pd.Series(labels_full).value_counts().sort_index()
    min_cluster_size = int(cluster_sizes.min())
    max_cluster_size = int(cluster_sizes.max())
    cluster_size_std = float(cluster_sizes.std(ddof=0))
    cluster_size_mean = float(cluster_sizes.mean())
    cluster_size_cv = float(cluster_size_std / cluster_size_mean) if cluster_size_mean else np.nan

    result = {
        "model": model_name,
        "feature_space": feature_space,
        "k": int(k),
        "train_time_sec": float(train_time_sec),
        "inertia": float(getattr(estimator, "inertia_", np.nan)),
        "silhouette": float(silhouette_score(X_eval, labels_sample)),
        "davies_bouldin": float(davies_bouldin_score(X_eval, labels_sample)),
        "calinski_harabasz": float(calinski_harabasz_score(X_eval, labels_sample)),
        "min_cluster_size": min_cluster_size,
        "max_cluster_size": max_cluster_size,
        "cluster_size_std": cluster_size_std,
        "cluster_size_cv": cluster_size_cv,
        "fit_note": fit_note,
        "fit_rows": int(fit_rows),
        "eval_rows": int(len(eval_idx)),
        "estimator": estimator,
        "labels_full": labels_full,
    }
    if fit_idx is not None:
        result["fit_sample_rows"] = int(len(fit_idx))
    return result


def run_experiments(
    artifacts: dict[str, object],
    k_values: list[int],
    eval_sample_size: int = 10000,
    random_state: int = 42,
) -> tuple[pd.DataFrame, list[dict[str, object]], np.ndarray]:
    eval_idx = make_eval_sample(artifacts["train_metadata"].shape[0], eval_sample_size, random_state=random_state)

    model_configs = [
        {
            "model_name": "KMeans",
            "feature_space": "svd",
            "estimator_cls": KMeans,
            "estimator_params": {
                "init": "k-means++",
                "n_init": 10,
                "max_iter": 300,
                "random_state": random_state,
            },
            "fit_sample_size": HEAVY_KMEANS_SVD_SAMPLE_SIZE,
        },
        {
            "model_name": "MiniBatchKMeans",
            "feature_space": "svd",
            "estimator_cls": MiniBatchKMeans,
            "estimator_params": {
                "init": "k-means++",
                "n_init": 10,
                "max_iter": 300,
                "batch_size": 4096,
                "random_state": random_state,
            },
            "fit_sample_size": None,
        },
        {
            "model_name": "KMeans",
            "feature_space": "umap",
            "estimator_cls": KMeans,
            "estimator_params": {
                "init": "k-means++",
                "n_init": 10,
                "max_iter": 300,
                "random_state": random_state,
            },
            "fit_sample_size": HEAVY_KMEANS_UMAP_SAMPLE_SIZE,
        },
    ]

    results: list[dict[str, object]] = []
    for config in model_configs:
        X_train = get_feature_matrix(artifacts, config["feature_space"], "train")
        X_eval = X_train[eval_idx]
        for k in k_values:
            print(f"Running {config['model_name']} on {config['feature_space']} with k={k} ...")
            result = evaluate_clustering(
                estimator_cls=config["estimator_cls"],
                estimator_params=config["estimator_params"],
                X_train=X_train,
                X_eval=X_eval,
                eval_idx=eval_idx,
                model_name=config["model_name"],
                feature_space=config["feature_space"],
                k=k,
                fit_sample_size=config["fit_sample_size"],
            )
            results.append(result)

    metrics_df = pd.DataFrame(
        [
            {key: value for key, value in result.items() if key not in {"estimator", "labels_full"}}
            for result in results
        ]
    )
    return metrics_df, results, eval_idx


def add_rank_columns(metrics_df: pd.DataFrame) -> pd.DataFrame:
    ranked = metrics_df.copy()
    ranked["rank_silhouette"] = ranked["silhouette"].rank(ascending=False, method="min")
    ranked["rank_davies_bouldin"] = ranked["davies_bouldin"].rank(ascending=True, method="min")
    ranked["rank_calinski_harabasz"] = ranked["calinski_harabasz"].rank(ascending=False, method="min")
    ranked["rank_cluster_balance"] = ranked["cluster_size_cv"].rank(ascending=True, method="min")
    ranked["total_rank"] = (
        ranked["rank_silhouette"]
        + ranked["rank_davies_bouldin"]
        + ranked["rank_calinski_harabasz"]
        + 0.5 * ranked["rank_cluster_balance"]
    )
    return ranked.sort_values(
        ["total_rank", "silhouette", "davies_bouldin", "calinski_harabasz"],
        ascending=[True, False, True, False],
    ).reset_index(drop=True)


def plot_metric(metrics_df: pd.DataFrame, metric: str, ylabel: str, filename: str, ascending: bool | None = None) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    for (model, feature_space), group in metrics_df.groupby(["model", "feature_space"]):
        group = group.sort_values("k")
        ax.plot(
            group["k"],
            group[metric],
            marker="o",
            linewidth=2,
            label=f"{model} - {feature_space}",
        )
    ax.set_title(f"{ylabel} theo số cụm k")
    ax.set_xlabel("Số cụm k")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_cluster_size_distribution(labels: np.ndarray, filename: str) -> None:
    counts = pd.Series(labels).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(counts.index.astype(str), counts.values, color="#4c78a8")
    ax.set_title("Phân phối kích thước cụm của mô hình tốt nhất")
    ax.set_xlabel("cluster_id")
    ax.set_ylabel("Số lượng mẫu")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_salary_by_cluster(train_clustered: pd.DataFrame, filename: str) -> None:
    salary_col = "salary_expected_million_vnd"
    if salary_col not in train_clustered.columns:
        return
    plot_df = train_clustered[[salary_col, "cluster_id"]].dropna()
    if plot_df.empty:
        return

    groups = [
        plot_df.loc[plot_df["cluster_id"] == cluster_id, salary_col].values
        for cluster_id in sorted(plot_df["cluster_id"].unique())
    ]
    labels = [str(cluster_id) for cluster_id in sorted(plot_df["cluster_id"].unique())]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.boxplot(groups, tick_labels=labels, showfliers=False)
    ax.set_title("Phân phối lương kỳ vọng theo cụm")
    ax.set_xlabel("cluster_id")
    ax.set_ylabel("salary_expected_million_vnd")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_umap_clusters(X_train_umap, labels: np.ndarray, filename: str, random_state: int = 42, max_points: int = 20000) -> None:
    sample_idx = make_eval_sample(X_train_umap.shape[0], max_points, random_state=random_state)
    X_plot = X_train_umap[sample_idx]
    y_plot = labels[sample_idx]
    if X_plot.shape[1] == 1:
        coords = np.column_stack([X_plot[:, 0], np.zeros(X_plot.shape[0])])
    else:
        coords = X_plot[:, :2]

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=y_plot,
        s=6,
        alpha=0.6,
        cmap="tab20",
    )
    ax.set_title("Minh họa phân cụm trên không gian UMAP 2D/sample")
    ax.set_xlabel("UMAP dim 1")
    ax.set_ylabel("UMAP dim 2")
    legend = ax.legend(*scatter.legend_elements(num=None), title="cluster_id", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.add_artist(legend)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)


def build_cluster_profiles(
    train_clustered: pd.DataFrame,
    tfidf_model,
    train_features,
) -> pd.DataFrame:
    profile_rows: list[dict[str, object]] = []
    total_rows = len(train_clustered)
    salary_col = "salary_expected_million_vnd"
    location_col = "location_simplified" if "location_simplified" in train_clustered.columns else ("location" if "location" in train_clustered.columns else None)

    for cluster_id, group in train_clustered.groupby("cluster_id", sort=True):
        row = {
            "cluster_id": int(cluster_id),
            "cluster_size": int(len(group)),
            "cluster_ratio": float(len(group) / total_rows),
            "salary_mean": float(group[salary_col].mean()) if salary_col in group.columns else np.nan,
            "salary_median": float(group[salary_col].median()) if salary_col in group.columns else np.nan,
            "salary_min": float(group[salary_col].min()) if salary_col in group.columns else np.nan,
            "salary_max": float(group[salary_col].max()) if salary_col in group.columns else np.nan,
            "top_job_title": top_values(group["job_title"]) if "job_title" in group.columns else "",
            "top_job_industry": top_values(group["job_industry"]) if "job_industry" in group.columns else "",
            "top_experience_level": top_values(group["experience_level"]) if "experience_level" in group.columns else "",
            "top_education_level": top_values(group["education_level"]) if "education_level" in group.columns else "",
            "top_job_position": top_values(group["job_position"]) if "job_position" in group.columns else "",
            "top_job_type": top_values(group["job_type"]) if "job_type" in group.columns else "",
            "top_location": top_values(group[location_col]) if location_col else "",
            "top_keywords": "",
            "suggested_cluster_label": "",
        }
        profile_rows.append(row)

    profiles = pd.DataFrame(profile_rows).sort_values("cluster_id").reset_index(drop=True)

    if train_features is not None and tfidf_model is not None:
        try:
            feature_names = np.asarray(tfidf_model.get_feature_names_out())
            labels = train_clustered["cluster_id"].to_numpy()
            keyword_texts = []
            for cluster_id in profiles["cluster_id"]:
                cluster_mask = labels == cluster_id
                cluster_matrix = train_features[cluster_mask]
                if cluster_matrix.shape[0] == 0:
                    keyword_texts.append("")
                    continue
                mean_tfidf = np.asarray(cluster_matrix.mean(axis=0)).ravel()
                top_idx = np.argsort(mean_tfidf)[-10:][::-1]
                top_terms = [feature_names[idx] for idx in top_idx if mean_tfidf[idx] > 0]
                keyword_texts.append("; ".join(top_terms))
            profiles["top_keywords"] = keyword_texts
        except Exception as exc:
            print(f"Không thể trích xuất top_keywords từ TF-IDF: {exc}")

    profiles["suggested_cluster_label"] = profiles.apply(build_suggested_label, axis=1)
    return profiles


def create_cluster_label_template(profiles: pd.DataFrame, output_path: Path, preserve_existing: bool = True) -> pd.DataFrame:
    template = profiles[
        [
            "cluster_id",
            "cluster_size",
            "top_job_industry",
            "top_job_title",
            "salary_mean",
            "suggested_cluster_label",
        ]
    ].copy()
    template["manual_cluster_label"] = ""

    if preserve_existing and output_path.exists():
        existing = pd.read_csv(output_path)
        if {"cluster_id", "manual_cluster_label"}.issubset(existing.columns):
            template = template.merge(
                existing[["cluster_id", "manual_cluster_label"]],
                on="cluster_id",
                how="left",
                suffixes=("", "_existing"),
            )
            template["manual_cluster_label"] = template["manual_cluster_label_existing"].fillna(template["manual_cluster_label"])
            template = template.drop(columns=["manual_cluster_label_existing"])

    template.to_csv(output_path, index=False)
    return template


def attach_cluster_labels(df: pd.DataFrame, label_template: pd.DataFrame) -> pd.DataFrame:
    label_map = label_template.set_index("cluster_id")["manual_cluster_label"].fillna("")
    suggested_map = label_template.set_index("cluster_id")["suggested_cluster_label"].fillna("")
    df = df.copy()
    df["cluster_label"] = df["cluster_id"].map(label_map)
    df["cluster_label"] = df["cluster_label"].where(df["cluster_label"].astype(str).str.strip() != "", df["cluster_id"].map(suggested_map))
    return df


def create_test_demo(test_clustered: pd.DataFrame, label_template: pd.DataFrame, random_state: int = 42, sample_size: int = 10) -> pd.DataFrame:
    labeled_test = attach_cluster_labels(test_clustered, label_template)
    keep_columns = [
        "job_title",
        "salary",
        "salary_expected_million_vnd",
        "job_industry",
        "experience_level",
        "education_level",
        "job_position",
        "job_type",
        "location",
        "location_simplified",
        "cluster_id",
        "cluster_label",
    ]
    available_columns = [column for column in keep_columns if column in labeled_test.columns]
    demo = labeled_test[available_columns].sample(n=min(sample_size, len(labeled_test)), random_state=random_state)
    return demo.reset_index(drop=True)


## 4.4. Thiết lập thực nghiệm

- `k_values = [3, 4, 5, 6, 7, 8, 10, 12]`
- `random_state = 42`
- `EVAL_SAMPLE_SIZE = 10000`
- `MiniBatchKMeans` dùng `batch_size = 4096`

Nếu cần kiểm tra nhanh pipeline, có thể chuyển `QUICK_MODE = True` để chỉ chạy `k = [3, 4]`. Notebook vẫn để mặc định đầy đủ `k_values` để phục vụ báo cáo chính thức.


In [ ]:
artifacts = load_artifacts(BASE_DIR)

display(Markdown("**Tóm tắt dữ liệu đầu vào**"))
input_summary = pd.DataFrame(
    [
        {"artifact": "X_train_svd", "shape": tuple(artifacts["train_svd"].shape)},
        {"artifact": "X_test_svd", "shape": tuple(artifacts["test_svd"].shape)},
        {"artifact": "X_train_umap", "shape": tuple(artifacts["train_umap"].shape)},
        {"artifact": "X_test_umap", "shape": tuple(artifacts["test_umap"].shape)},
        {"artifact": "train_metadata", "shape": tuple(artifacts["train_metadata"].shape)},
        {"artifact": "test_metadata", "shape": tuple(artifacts["test_metadata"].shape)},
        {"artifact": "X_train_features", "shape": None if artifacts["train_features"] is None else tuple(artifacts["train_features"].shape)},
    ]
)
display(input_summary)

if artifacts["train_features"] is None:
    print("Lưu ý: không tìm thấy X_train_features.npz nên notebook sẽ bỏ qua bước top_keywords từ TF-IDF.")


## 4.5. Huấn luyện và đánh giá

Các metric nội tại được tính như sau:

- **Inertia**: càng thấp càng tốt, phản ánh độ chặt của cụm với KMeans-family.
- **Silhouette**: càng cao càng tốt, phản ánh mức tách biệt giữa các cụm.
- **Davies-Bouldin**: càng thấp càng tốt.
- **Calinski-Harabasz**: càng cao càng tốt.

Do dữ liệu lớn, ba metric nội tại ngoài Inertia được tính trên sample train cố định để tránh chi phí `O(n^2)` quá lớn.


In [ ]:
metrics_df, experiment_results, eval_idx = run_experiments(
    artifacts=artifacts,
    k_values=K_VALUES,
    eval_sample_size=EVAL_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

metrics_ranked = add_rank_columns(metrics_df)
metrics_ranked.to_csv(CLUSTERING_DIR / "clustering_metrics.csv", index=False)

display(metrics_ranked.head(10))


## 4.6. So sánh mô hình và chọn số cụm

Notebook dùng rank tổng hợp để chọn mô hình tốt nhất:

- `rank_silhouette`: Silhouette giảm dần
- `rank_davies_bouldin`: Davies-Bouldin tăng dần
- `rank_calinski_harabasz`: Calinski-Harabasz giảm dần
- `rank_cluster_balance`: `cluster_size_cv` tăng dần, chỉ đóng vai trò nhẹ

Lựa chọn cuối cùng vẫn nên cân nhắc thêm tính diễn giải của cụm, không chỉ dựa máy móc vào metric.


In [ ]:
plot_metric(metrics_ranked, "inertia", "Inertia", "inertia_by_k.png")
plot_metric(metrics_ranked, "silhouette", "Silhouette", "silhouette_by_k.png")
plot_metric(metrics_ranked, "davies_bouldin", "Davies-Bouldin", "davies_bouldin_by_k.png")
plot_metric(metrics_ranked, "calinski_harabasz", "Calinski-Harabasz", "calinski_harabasz_by_k.png")

best_row = metrics_ranked.iloc[0]
best_key = (
    best_row["model"],
    best_row["feature_space"],
    int(best_row["k"]),
)
best_result = next(
    result
    for result in experiment_results
    if (result["model"], result["feature_space"], int(result["k"])) == best_key
)

print("Best model:", best_key)
display(best_row.to_frame().T)


In [ ]:
best_feature_space = best_result["feature_space"]
X_test_best = artifacts[f"test_{best_feature_space}"]
train_labels = best_result["labels_full"]
test_labels = best_result["estimator"].predict(X_test_best)

train_clustered = artifacts["train_metadata"].copy()
test_clustered = artifacts["test_metadata"].copy()
train_clustered["cluster_id"] = train_labels
test_clustered["cluster_id"] = test_labels

train_clustered.to_csv(CLUSTERING_DIR / "train_clustered.csv", index=False)
test_clustered.to_csv(CLUSTERING_DIR / "test_clustered.csv", index=False)

best_model_payload = {
    "model_name": best_result["model"],
    "feature_space": best_result["feature_space"],
    "k": int(best_result["k"]),
    "estimator": best_result["estimator"],
    "fit_note": best_result["fit_note"],
    "fit_rows": int(best_result["fit_rows"]),
    "random_state": RANDOM_STATE,
}
joblib.dump(best_model_payload, CLUSTERING_DIR / "best_model.joblib")

best_model_info = {
    "model": best_result["model"],
    "feature_space": best_result["feature_space"],
    "k": int(best_result["k"]),
    "train_time_sec": float(best_result["train_time_sec"]),
    "inertia": float(best_result["inertia"]),
    "silhouette": float(best_result["silhouette"]),
    "davies_bouldin": float(best_result["davies_bouldin"]),
    "calinski_harabasz": float(best_result["calinski_harabasz"]),
    "cluster_size_cv": float(best_result["cluster_size_cv"]),
    "fit_note": best_result["fit_note"],
    "fit_rows": int(best_result["fit_rows"]),
    "eval_rows": int(best_result["eval_rows"]),
    "k_values_tested": K_VALUES,
    "evaluation_sample_size": EVAL_SAMPLE_SIZE,
    "selection_rule": "total_rank = rank_silhouette + rank_davies_bouldin + rank_calinski_harabasz + 0.5 * rank_cluster_balance",
    "test_usage_note": "Tập test chỉ dùng để predict cluster_id và tạo demo 10 mẫu, không dùng để chọn mô hình.",
}
with (CLUSTERING_DIR / "best_model_info.json").open("w", encoding="utf-8") as f:
    json.dump(best_model_info, f, ensure_ascii=False, indent=2)


## 4.7. Phân tích đặc điểm từng cụm

Phần này tạo `cluster_profiles.csv` để tóm tắt quy mô cụm, phân phối lương, nhóm ngành nghề/chức danh phổ biến và từ khóa đại diện nếu có đủ artifact TF-IDF train.


In [ ]:
cluster_profiles = build_cluster_profiles(
    train_clustered=train_clustered,
    tfidf_model=artifacts["tfidf_model"],
    train_features=artifacts["train_features"],
)
cluster_profiles.to_csv(CLUSTERING_DIR / "cluster_profiles.csv", index=False)

cluster_label_template = create_cluster_label_template(
    profiles=cluster_profiles,
    output_path=CLUSTERING_DIR / "cluster_label_template.csv",
    preserve_existing=PRESERVE_EXISTING_MANUAL_LABELS,
)

plot_cluster_size_distribution(train_labels, "cluster_size_distribution.png")
plot_salary_by_cluster(train_clustered, "salary_by_cluster.png")
plot_umap_clusters(artifacts["train_umap"], train_labels, "umap_2d_clusters.png", random_state=RANDOM_STATE, max_points=VIS_SAMPLE_SIZE)

display(cluster_profiles.head())


## 4.8. Gán nhãn cụm thủ công

Nhãn cụm cuối cùng được gán thủ công dựa trên:

- top ngành nghề phổ biến trong từng cụm
- top chức danh công việc
- mức lương trung bình/trung vị
- kinh nghiệm và đặc điểm phổ biến trong `cluster_profiles.csv`

Ở bước này notebook **không huấn luyện lại mô hình**. Notebook chỉ đọc lại các artifact đã có, sau đó cập nhật `manual_cluster_label` và `final_cluster_label` để hoàn thiện phần diễn giải cụm cho báo cáo.


In [ ]:
cluster_label_map = {
    0: "Bán lẻ - dịch vụ - lao động phổ thông",
    1: "Chăm sóc khách hàng - tư vấn giáo dục",
    2: "Kỹ thuật - sản xuất - bảo trì",
    3: "Kinh doanh - bán hàng - phát triển thị trường",
    4: "Chuyên viên kinh doanh - phát triển khách hàng",
    5: "Kế toán - kiểm toán - tài chính doanh nghiệp",
    6: "Hành chính nhân sự - văn phòng - xuất nhập khẩu",
    7: "Xây dựng - kiến trúc - kỹ thuật công trình",
    8: "Bán hàng cửa hàng - quản lý khu vực",
    9: "Telesales - tư vấn khách hàng",
    10: "Tài chính - ngân hàng - tư vấn pháp lý",
    11: "Marketing - truyền thông - thiết kế",
}

cluster_files = {
    "profiles": CLUSTERING_DIR / "cluster_profiles.csv",
    "label_template": CLUSTERING_DIR / "cluster_label_template.csv",
    "train_clustered": CLUSTERING_DIR / "train_clustered.csv",
    "test_clustered": CLUSTERING_DIR / "test_clustered.csv",
}
missing_cluster_files = [str(path) for path in cluster_files.values() if not path.exists()]
if missing_cluster_files:
    raise FileNotFoundError("Thiếu artifact clustering để gán nhãn cụm: " + ", ".join(missing_cluster_files))

cluster_profiles = pd.read_csv(cluster_files["profiles"])
cluster_label_template = pd.read_csv(cluster_files["label_template"])
train_clustered = pd.read_csv(cluster_files["train_clustered"])
test_clustered = pd.read_csv(cluster_files["test_clustered"])

cluster_profiles["manual_cluster_label"] = cluster_profiles["cluster_id"].map(cluster_label_map)
cluster_profiles["final_cluster_label"] = cluster_profiles["manual_cluster_label"]

cluster_label_template["manual_cluster_label"] = cluster_label_template["cluster_id"].map(cluster_label_map)

train_clustered["cluster_label"] = train_clustered["cluster_id"].map(cluster_label_map)
test_clustered["cluster_label"] = test_clustered["cluster_id"].map(cluster_label_map)

cluster_label_template.to_csv(cluster_files["label_template"], index=False)
cluster_profiles.to_csv(cluster_files["profiles"], index=False)
train_clustered.to_csv(cluster_files["train_clustered"], index=False)
test_clustered.to_csv(cluster_files["test_clustered"], index=False)

display(cluster_profiles[["cluster_id", "manual_cluster_label", "cluster_size", "salary_mean", "salary_median"]].sort_values("cluster_id"))


## 4.9. Demo 10 mẫu trên tập kiểm thử

Tập test không được dùng để chọn mô hình. Sau khi đã có nhãn cụm thủ công, notebook chỉ dùng `test_clustered.csv` để:

- gắn `cluster_label`
- lấy ngẫu nhiên 10 mẫu minh họa với `random_state = 42`
- lưu lại `test_demo_10_samples.csv` phục vụ báo cáo


In [ ]:
demo_columns = [
    "job_title",
    "salary",
    "salary_expected_million_vnd",
    "job_industry",
    "experience_level",
    "education_level",
    "job_position",
    "job_type",
    "location",
    "location_simplified",
    "cluster_id",
    "cluster_label",
]
available_demo_columns = [column for column in demo_columns if column in test_clustered.columns]
test_demo = (
    test_clustered[available_demo_columns]
    .sample(n=min(10, len(test_clustered)), random_state=42)
    .reset_index(drop=True)
)
test_demo.to_csv(CLUSTERING_DIR / "test_demo_10_samples.csv", index=False)

display(test_demo)

valid_cluster_ids = set(cluster_label_map)
assert set(train_clustered["cluster_id"].unique()).issubset(valid_cluster_ids)
assert set(test_clustered["cluster_id"].unique()).issubset(valid_cluster_ids)
assert train_clustered["cluster_label"].notna().all()
assert test_clustered["cluster_label"].notna().all()
assert (train_clustered["cluster_label"].astype(str).str.strip() != "").all()
assert (test_clustered["cluster_label"].astype(str).str.strip() != "").all()
assert cluster_profiles.shape[0] == 12
assert test_demo.shape[0] == 10


## 4.10. Kết luận phần mô hình hóa

Mô hình cuối cùng được sử dụng là **KMeans trên không gian UMAP với k = 12**. Theo kết quả đánh giá trên sample 10.000 mẫu train, mô hình này đạt Silhouette khoảng **0.3660**, Davies-Bouldin khoảng **1.0425** và Calinski-Harabasz khoảng **2177.10**.

Tập test **không** được dùng để chọn mô hình hay chọn số cụm. Test chỉ được dùng sau cùng để gán `cluster_id`, gắn `cluster_label` và tạo bảng demo 10 mẫu minh họa.

Sau khi phân tích `cluster_profiles.csv`, các cụm đã được diễn giải và gán nhãn thủ công để phản ánh rõ hơn các nhóm việc làm phổ biến tại Việt Nam. Bộ artifact cuối cùng có thể dùng trực tiếp cho phần viết báo cáo và phân tích kết quả clustering.


In [ ]:
summary_paths = [
    CLUSTERING_DIR / "cluster_label_template.csv",
    CLUSTERING_DIR / "cluster_profiles.csv",
    CLUSTERING_DIR / "train_clustered.csv",
    CLUSTERING_DIR / "test_clustered.csv",
    CLUSTERING_DIR / "test_demo_10_samples.csv",
]
for path in summary_paths:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu file đầu ra sau khi cập nhật nhãn cụm: {path}")

print("Đã cập nhật xong nhãn cụm thủ công và demo test.")


## 4.11. Tối ưu UMAP và mở rộng tìm kiếm số cụm k

Kết quả baseline tốt nhất hiện tại là **KMeans trên UMAP với k = 12**. Vì Silhouette tăng mạnh ở các k lớn và đạt giá trị cao tại `k=12`, nhóm thử thêm các giá trị lân cận như `13, 14, 15, 16, 18, 20` để kiểm tra khả năng cải thiện.

Vì UMAP ảnh hưởng mạnh đến cấu trúc embedding và ranh giới giữa các cụm, nhóm thử thêm một số cấu hình `n_neighbors`, `min_dist` và `n_components`. Đồng thời, nhóm bổ sung phân tích **Inertia/Elbow** để quan sát tốc độ giảm sai số trong cụm khi tăng `k`.

Tuy nhiên, Inertia không được dùng một mình để quyết định vì chỉ số này thường giảm khi tăng số cụm. Quyết định cuối cùng dựa trên tổng hợp:

- Silhouette càng cao càng tốt
- Davies-Bouldin càng thấp càng tốt
- Calinski-Harabasz càng cao càng tốt
- Inertia/Elbow hợp lý
- kích thước cụm không quá lệch
- khả năng diễn giải cụm trong ngữ cảnh thị trường việc làm Việt Nam

Do dữ liệu lớn, phần tuning này dùng `TUNING_FIT_SAMPLE_SIZE = 50000` và `EVAL_SAMPLE_SIZE = 10000` trên train để đảm bảo runtime ổn định. Test **không** tham gia chọn UMAP config, chọn `k` hay chọn mô hình.


In [ ]:
TUNING_METRICS_PATH = CLUSTERING_DIR / "umap_tuning_metrics.csv"
TUNING_SUMMARY_PATH = CLUSTERING_DIR / "umap_tuning_summary.md"
TUNING_BEST_CONFIG_PATH = CLUSTERING_DIR / "umap_tuning_best_config.json"

for path in [TUNING_METRICS_PATH, TUNING_SUMMARY_PATH, TUNING_BEST_CONFIG_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu output tuning: {path}")

umap_tuning_metrics = pd.read_csv(TUNING_METRICS_PATH)
umap_tuning_ok = umap_tuning_metrics.loc[umap_tuning_metrics["status"] == "ok"].copy()
assert not umap_tuning_ok.empty, "Không có cấu hình tuning nào chạy thành công."

best_tuned_row = umap_tuning_ok.sort_values(["total_rank", "silhouette"], ascending=[True, False]).iloc[0]
display(umap_tuning_ok.sort_values(["total_rank", "silhouette"], ascending=[True, False]).head(10))

with TUNING_BEST_CONFIG_PATH.open(encoding="utf-8") as f:
    tuning_best_config = json.load(f)
tuning_best_config


## 4.12. Kết luận sau tuning

Best tuned config hiện tại là **input_variant = svd_l2norm**, `n_neighbors = 50`, `min_dist = 0.0`, `n_components = 10`, `k = 13`.

So với baseline:

- `silhouette_gain = 0.1046`
- `db_gain = 0.2557`
- `ch_gain = 1539.01`

Nhóm giữ cấu hình baseline KMeans + UMAP + k=12 vì tuning không cải thiện đủ đáng kể so với baseline hoặc làm tăng độ phức tạp diễn giải.

Trong mọi trường hợp, tập test vẫn chỉ được dùng để gán `cluster_id` sau khi đã chốt mô hình, không được dùng để chọn UMAP config hay chọn số cụm.


In [ ]:
required_tuning_outputs = [
    CLUSTERING_DIR / "umap_tuning_metrics.csv",
    CLUSTERING_DIR / "umap_tuning_best_config.json",
    CLUSTERING_DIR / "umap_tuning_summary.md",
    FIGURES_DIR / "tuning_inertia_elbow_by_k.png",
    FIGURES_DIR / "tuning_silhouette_by_k.png",
    FIGURES_DIR / "tuning_davies_bouldin_by_k.png",
    FIGURES_DIR / "tuning_calinski_harabasz_by_k.png",
]
for path in required_tuning_outputs:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu output tuning: {path}")

print("Đã hoàn tất phần tuning UMAP và mở rộng tìm kiếm k.")


## 4.13. Kiểm tra tuned final candidate

Kết quả tuning cho thấy cấu hình `svd_l2norm + UMAP(n_neighbors=50, min_dist=0.0, n_components=10, metric='cosine') + KMeans(k=13)` cải thiện metric nội tại rõ rệt so với baseline. Tuy nhiên trước khi thay mô hình cuối, nhóm cần kiểm tra khả năng diễn giải của 13 cụm.

Ở bước này notebook chỉ dùng **train** để fit UMAP và KMeans. Tập **test** chỉ được transform bằng tuned UMAP và predict bằng tuned KMeans sau khi cấu hình đã được cố định từ bước tuning.


In [ ]:
tuned_output_paths = [
    CLUSTERING_DIR / "tuned_umap_model.joblib",
    CLUSTERING_DIR / "tuned_best_model.joblib",
    CLUSTERING_DIR / "tuned_best_model_info.json",
    CLUSTERING_DIR / "tuned_train_clustered.csv",
    CLUSTERING_DIR / "tuned_test_clustered.csv",
    CLUSTERING_DIR / "tuned_cluster_profiles.csv",
    CLUSTERING_DIR / "tuned_cluster_label_template.csv",
    CLUSTERING_DIR / "tuned_test_demo_10_samples.csv",
    CLUSTERING_DIR / "tuned_final_candidate_summary.md",
]
for path in tuned_output_paths:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu tuned candidate output: {path}")

tuned_cluster_profiles = pd.read_csv(CLUSTERING_DIR / "tuned_cluster_profiles.csv")
tuned_test_demo = pd.read_csv(CLUSTERING_DIR / "tuned_test_demo_10_samples.csv")

assert tuned_cluster_profiles.shape[0] == 13
assert tuned_test_demo.shape[0] == 10

display(tuned_cluster_profiles[[
    "cluster_id",
    "cluster_size",
    "salary_mean",
    "salary_median",
    "top_job_industry",
    "top_job_title",
    "suggested_cluster_label",
]])
display(tuned_test_demo)


## 4.14. Kết luận cuối cùng

Sau khi tuning và kiểm tra profile cụm, nhóm lựa chọn **tuned KMeans + UMAP với k=13** làm mô hình cuối cùng.

Mô hình tuned cải thiện rõ rệt các metric nội tại so với baseline, đồng thời 13 cụm vẫn có thể diễn giải được theo ngành nghề, chức danh, mức lương và đặc trưng nghiệp vụ. Baseline k=12 vẫn được giữ lại trong thư mục artifact để phục vụ so sánh, nhưng mô hình chính dùng cho báo cáo cuối là tuned candidate.


In [ ]:
required_tuned_figures = [
    FIGURES_DIR / "tuned_cluster_size_distribution.png",
    FIGURES_DIR / "tuned_salary_by_cluster.png",
    FIGURES_DIR / "tuned_umap_2d_clusters.png",
]
for path in required_tuned_figures:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu tuned candidate figure: {path}")

required_final_outputs = [
    CLUSTERING_DIR / "final_best_model_info.json",
    CLUSTERING_DIR / "final_cluster_profiles.csv",
    CLUSTERING_DIR / "final_train_clustered.csv",
    CLUSTERING_DIR / "final_test_clustered.csv",
    CLUSTERING_DIR / "final_test_demo_10_samples.csv",
    CLUSTERING_DIR / "final_modeling_report_summary.md",
]
for path in required_final_outputs:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu final model output: {path}")

print("Đã chốt tuned KMeans + UMAP + k=13 làm mô hình cuối cùng.")


## 4.15. Thử nghiệm UMAP fit trên toàn bộ tập huấn luyện

Nhóm thực hiện thêm thí nghiệm fit UMAP trên **toàn bộ tập train 90%** sau khi chuẩn hóa L2 để kiểm tra xem mô hình có thể cải thiện thêm hoặc giữ chất lượng tương đương nhưng đáp ứng chặt chẽ hơn yêu cầu huấn luyện trên train hay không.

Trong thí nghiệm này:

- chỉ fit UMAP và KMeans trên train
- test chỉ được transform bằng UMAP đã fit từ train
- test chỉ dùng để predict và demo, không dùng để chọn mô hình


In [ ]:
full_umap_outputs = [
    CLUSTERING_DIR / "full_umap_model.joblib",
    CLUSTERING_DIR / "full_umap_kmeans_model.joblib",
    CLUSTERING_DIR / "full_umap_best_model_info.json",
    CLUSTERING_DIR / "full_umap_train_clustered.csv",
    CLUSTERING_DIR / "full_umap_test_clustered.csv",
    CLUSTERING_DIR / "full_umap_cluster_profiles.csv",
    CLUSTERING_DIR / "full_umap_cluster_label_template.csv",
    CLUSTERING_DIR / "full_umap_test_demo_10_samples.csv",
    CLUSTERING_DIR / "full_umap_comparison_summary.md",
]
for path in full_umap_outputs:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu output full-train UMAP: {path}")

full_umap_profiles = pd.read_csv(CLUSTERING_DIR / "full_umap_cluster_profiles.csv")
full_umap_demo = pd.read_csv(CLUSTERING_DIR / "full_umap_test_demo_10_samples.csv")
assert full_umap_profiles.shape[0] == 13
assert full_umap_demo.shape[0] == 10
display(full_umap_profiles[[
    "cluster_id",
    "cluster_size",
    "salary_mean",
    "salary_median",
    "top_job_industry",
    "top_job_title",
    "suggested_cluster_label",
]])
display(full_umap_demo)


## 4.16. Kết luận thí nghiệm full-train UMAP

Kết quả thực nghiệm cho thấy full-train UMAP **chạy thành công**, nhưng chất lượng phân cụm không tốt bằng mô hình hiện tại dùng UMAP fit trên 100000 train đại diện. Silhouette giảm, Davies-Bouldin tăng và Calinski-Harabasz giảm rõ rệt, dù phân phối kích thước cụm cân bằng hơn.

Vì vậy nhóm **không khuyến nghị chuyển sang full-train UMAP** ở thời điểm này. Mô hình cuối vẫn giữ là tuned KMeans + UMAP + k=13 với UMAP fit trên 100000 train đại diện, do cấu hình đó cho chất lượng cụm tốt hơn trên cùng quy trình đánh giá train-only.


In [ ]:
required_full_umap_figures = [
    FIGURES_DIR / "full_umap_cluster_size_distribution.png",
    FIGURES_DIR / "full_umap_salary_by_cluster.png",
    FIGURES_DIR / "full_umap_2d_clusters.png",
]
for path in required_full_umap_figures:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu full-train UMAP figure: {path}")

print("Đã hoàn tất thí nghiệm full-train UMAP và lưu toàn bộ output.")
